# Business Impact & Storytelling

## Supply Chain Late Delivery Prediction

---

### Executive Summary

This notebook translates ML model performance into **business value** for supply chain stakeholders.

### Key Questions Answered

1. **How much money can we save?** - Cost-benefit analysis
2. **What actions can we take?** - Intervention strategies
3. **Who should use this?** - Deployment recommendations
4. **What's next?** - Future improvements

---

### Presentation Flow (10 minutes)

| Section | Time | Focus |
|---------|------|-------|
| Problem Statement | 1 min | Why late deliveries matter |
| Solution Overview | 2 min | ML approach and features |
| Model Performance | 2 min | Key metrics and what they mean |
| Business Impact | 3 min | ROI and cost savings |
| Recommendations | 2 min | Next steps and deployment |

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import plotly.io as pio
pio.templates.default = "plotly_white"

print("Libraries loaded")

In [ ]:
# Load model and data
from src.data.preprocess import load_or_preprocess
from src.features.build_features import build_features_pipeline

# Load data
df = load_or_preprocess()
X, y = build_features_pipeline(df)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Load best model
model_dir = Path('../models')
model_files = sorted(model_dir.glob('best_model*.pkl'))

if model_files:
    best_model = joblib.load(model_files[-1])
    print(f"Model loaded: {model_files[-1].name}")
    MODEL_LOADED = True
else:
    print("No trained model found. Run 04_model_training.ipynb first.")
    MODEL_LOADED = False
    # Train a quick model for demo purposes
    from sklearn.ensemble import RandomForestClassifier
    best_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    best_model.fit(X_train, y_train)
    print("Trained demo model for analysis")
    MODEL_LOADED = True

In [ ]:
# Generate predictions
if MODEL_LOADED:
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Test set: {len(y_test):,} orders")
    print(f"Model accuracy: {accuracy:.1%}")

---

## 1. The Problem: Late Deliveries Cost Money

**Talking Point**: "Late deliveries aren't just operational issues - they directly impact revenue, customer loyalty, and brand reputation."

In [ ]:
# ============================================================
# PROBLEM QUANTIFICATION
# ============================================================

# Calculate baseline statistics
total_orders = len(y)
late_orders = y.sum()
late_rate = y.mean() * 100

# Business cost assumptions (configurable)
COST_PER_LATE_DELIVERY = 75  # Customer service, refunds, reputation damage
COST_PER_INTERVENTION = 15   # Proactive shipping upgrade, communication
REVENUE_SAVED_PER_CATCH = 50  # Revenue retained by preventing late delivery

# Current state (no ML)
current_late_cost = late_orders * COST_PER_LATE_DELIVERY

print("THE LATE DELIVERY PROBLEM")
print("=" * 60)
print(f"\nDataset Overview:")
print(f"   Total orders: {total_orders:,}")
print(f"   Late deliveries: {late_orders:,} ({late_rate:.1f}%)")
print(f"\nCost Assumptions:")
print(f"   Cost per late delivery: ${COST_PER_LATE_DELIVERY}")
print(f"   Cost per proactive intervention: ${COST_PER_INTERVENTION}")
print(f"   Revenue saved per caught late delivery: ${REVENUE_SAVED_PER_CATCH}")
print(f"\nCurrent Annual Cost (estimated):")
print(f"   ${current_late_cost:,.0f} in late delivery damages")

In [ ]:
# Visualize the problem
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "indicator"}]],
    subplot_titles=('<b>Delivery Status Distribution</b>', '<b>Annual Late Delivery Cost</b>')
)

# Pie chart
fig.add_trace(
    go.Pie(
        labels=['On-Time', 'Late'],
        values=[total_orders - late_orders, late_orders],
        hole=0.4,
        marker_colors=['#2ecc71', '#e74c3c'],
        textinfo='percent+label',
        textfont_size=14
    ),
    row=1, col=1
)

# Cost indicator
fig.add_trace(
    go.Indicator(
        mode="number",
        value=current_late_cost,
        number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 50, 'color': '#e74c3c'}},
        title={'text': 'Estimated Annual Cost'}
    ),
    row=1, col=2
)

fig.update_layout(
    height=400,
    title='<b>The Late Delivery Problem</b>'
)
fig.show()

---

## 2. The Solution: Predictive ML Model

**Talking Point**: "Our ML model predicts late deliveries BEFORE they happen, enabling proactive interventions."

In [ ]:
# ============================================================
# MODEL PERFORMANCE SUMMARY
# ============================================================

print("ML MODEL PERFORMANCE")
print("=" * 60)
print(f"\nConfusion Matrix Results (Test Set: {len(y_test):,} orders):")
print(f"   True Positives (Late caught): {tp:,}")
print(f"   True Negatives (On-time correct): {tn:,}")
print(f"   False Positives (False alarms): {fp:,}")
print(f"   False Negatives (Missed late): {fn:,}")
print(f"\nKey Metrics:")
print(f"   Accuracy: {accuracy:.1%}")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%} (late deliveries caught)")
print(f"   F1 Score: {f1:.3f}")

In [ ]:
# Visualize model results
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=('Late Deliveries Caught', 'False Alarm Rate', 'Overall Accuracy')
)

# Recall (late caught)
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=recall * 100,
        number={'suffix': '%', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#2ecc71'},
            'steps': [
                {'range': [0, 50], 'color': '#fadbd8'},
                {'range': [50, 75], 'color': '#fdebd0'},
                {'range': [75, 100], 'color': '#d5f5e3'}
            ],
            'threshold': {'line': {'color': 'red', 'width': 4}, 'value': 80}
        }
    ),
    row=1, col=1
)

# False positive rate
fpr_rate = fp / (fp + tn) * 100 if (fp + tn) > 0 else 0
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=fpr_rate,
        number={'suffix': '%', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 50]},
            'bar': {'color': '#e74c3c'},
            'steps': [
                {'range': [0, 10], 'color': '#d5f5e3'},
                {'range': [10, 25], 'color': '#fdebd0'},
                {'range': [25, 50], 'color': '#fadbd8'}
            ]
        }
    ),
    row=1, col=2
)

# Accuracy
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=accuracy * 100,
        number={'suffix': '%', 'font': {'size': 40}},
        gauge={
            'axis': {'range': [0, 100]},
            'bar': {'color': '#3498db'},
            'steps': [
                {'range': [0, 60], 'color': '#fadbd8'},
                {'range': [60, 80], 'color': '#fdebd0'},
                {'range': [80, 100], 'color': '#d5f5e3'}
            ]
        }
    ),
    row=1, col=3
)

fig.update_layout(
    height=350,
    title='<b>Model Performance Dashboard</b>'
)
fig.show()

---

## 3. Business Impact: ROI Calculation

**Talking Point**: "Let me show you the financial impact - this model can save significant money through proactive interventions."

In [ ]:
# ============================================================
# ROI CALCULATION
# ============================================================

# Scale to annual estimate (assuming test set represents typical distribution)
scale_factor = total_orders / len(y_test)

# SCENARIO 1: No ML (Current State)
scenario1_late_cost = late_orders * COST_PER_LATE_DELIVERY
scenario1_intervention_cost = 0
scenario1_total = scenario1_late_cost

# SCENARIO 2: With ML Model
# Caught late deliveries - we intervene (upgrade shipping, notify customer)
caught_late = tp * scale_factor
# Missed late deliveries - still incur cost
missed_late = fn * scale_factor
# False alarms - unnecessary intervention cost
false_alarms = fp * scale_factor

scenario2_late_cost = missed_late * COST_PER_LATE_DELIVERY
scenario2_intervention_cost = (caught_late + false_alarms) * COST_PER_INTERVENTION
scenario2_saved = caught_late * REVENUE_SAVED_PER_CATCH
scenario2_total = scenario2_late_cost + scenario2_intervention_cost - scenario2_saved

# Net savings
net_savings = scenario1_total - scenario2_total
roi_pct = (net_savings / scenario1_total) * 100 if scenario1_total > 0 else 0

print("ROI CALCULATION")
print("=" * 60)
print(f"\nSCENARIO 1: Without ML (Current State)")
print(f"   Late delivery costs: ${scenario1_late_cost:,.0f}")
print(f"   Total cost: ${scenario1_total:,.0f}")

print(f"\nSCENARIO 2: With ML Model")
print(f"   Late deliveries caught: {caught_late:,.0f}")
print(f"   Late deliveries missed: {missed_late:,.0f}")
print(f"   False alarms: {false_alarms:,.0f}")
print(f"   \n   Remaining late delivery cost: ${scenario2_late_cost:,.0f}")
print(f"   Intervention costs: ${scenario2_intervention_cost:,.0f}")
print(f"   Revenue saved: ${scenario2_saved:,.0f}")
print(f"   Total cost: ${scenario2_total:,.0f}")

print(f"\nNET SAVINGS: ${net_savings:,.0f} ({roi_pct:.1f}% reduction)")

In [ ]:
# Visualize ROI
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "indicator"}]],
    subplot_titles=('<b>Cost Comparison</b>', '<b>Annual Savings</b>')
)

# Cost comparison bar chart
fig.add_trace(
    go.Bar(
        x=['Without ML', 'With ML'],
        y=[scenario1_total, max(0, scenario2_total)],
        marker_color=['#e74c3c', '#2ecc71'],
        text=[f'${scenario1_total:,.0f}', f'${max(0, scenario2_total):,.0f}'],
        textposition='outside',
        textfont={'size': 16}
    ),
    row=1, col=1
)

# Savings indicator
fig.add_trace(
    go.Indicator(
        mode="number+delta",
        value=net_savings,
        number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 50, 'color': '#2ecc71'}},
        delta={'reference': 0, 'position': 'bottom'},
        title={'text': f'{roi_pct:.0f}% Cost Reduction'}
    ),
    row=1, col=2
)

fig.update_layout(
    height=400,
    title='<b>Business Impact: Return on Investment</b>',
    showlegend=False
)
fig.update_yaxes(title_text='Cost ($)', row=1, col=1)
fig.show()

---

## 4. Intervention Strategy

**Talking Point**: "Here's how operations teams can use these predictions to take action."

In [ ]:
# ============================================================
# INTERVENTION STRATEGY BY RISK TIER
# ============================================================

# Create risk tiers based on probability
risk_tiers = pd.DataFrame({
    'probability': y_proba,
    'actual': y_test.values
})

risk_tiers['risk_tier'] = pd.cut(
    risk_tiers['probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
)

# Calculate stats by tier
tier_stats = risk_tiers.groupby('risk_tier').agg({
    'probability': 'count',
    'actual': 'mean'
}).rename(columns={'probability': 'count', 'actual': 'late_rate'})
tier_stats['late_rate'] = tier_stats['late_rate'] * 100

# Recommended actions
actions = {
    'Low Risk': 'Standard processing - no action needed',
    'Medium Risk': 'Monitor shipment - alert if delayed',
    'High Risk': 'Proactive customer notification',
    'Critical': 'Upgrade shipping + notify customer + alert ops'
}

print("INTERVENTION STRATEGY BY RISK TIER")
print("=" * 70)
print(f"\n{'Risk Tier':<15} {'Orders':<12} {'Actual Late %':<15} {'Action'}")
print("-" * 70)
for tier in ['Low Risk', 'Medium Risk', 'High Risk', 'Critical']:
    if tier in tier_stats.index:
        row = tier_stats.loc[tier]
        print(f"{tier:<15} {int(row['count']):<12,} {row['late_rate']:<15.1f} {actions[tier]}")

In [ ]:
# Visualize risk tiers
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "bar"}]],
    subplot_titles=('<b>Orders by Risk Tier</b>', '<b>Actual Late Rate by Tier</b>')
)

tier_colors = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12', 
               'High Risk': '#e67e22', 'Critical': '#e74c3c'}

tiers_ordered = ['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
tier_stats_ordered = tier_stats.reindex(tiers_ordered).dropna()

# Orders by tier
fig.add_trace(
    go.Bar(
        x=tier_stats_ordered.index,
        y=tier_stats_ordered['count'],
        marker_color=[tier_colors.get(t, '#7f8c8d') for t in tier_stats_ordered.index],
        text=[f"{int(v):,}" for v in tier_stats_ordered['count']],
        textposition='outside'
    ),
    row=1, col=1
)

# Late rate by tier
fig.add_trace(
    go.Bar(
        x=tier_stats_ordered.index,
        y=tier_stats_ordered['late_rate'],
        marker_color=[tier_colors.get(t, '#7f8c8d') for t in tier_stats_ordered.index],
        text=[f"{v:.0f}%" for v in tier_stats_ordered['late_rate']],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(
    height=400,
    title='<b>Risk-Based Intervention Strategy</b>',
    showlegend=False
)
fig.update_yaxes(title_text='Number of Orders', row=1, col=1)
fig.update_yaxes(title_text='Actual Late Rate (%)', row=1, col=2)
fig.show()

---

## 5. Key Insights & Recommendations

**Talking Point**: "Based on our analysis, here are the actionable insights and next steps."

In [ ]:
# ============================================================
# EXECUTIVE SUMMARY DASHBOARD
# ============================================================

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=(
        'Total Orders Analyzed', 'Late Delivery Rate', 'Model Accuracy',
        'Late Deliveries Caught', 'Annual Savings', 'ROI'
    )
)

# Row 1
fig.add_trace(go.Indicator(
    mode="number", value=total_orders,
    number={'valueformat': ',', 'font': {'size': 36, 'color': '#3498db'}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=late_rate,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#e74c3c'}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=accuracy * 100,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#2ecc71'}}
), row=1, col=3)

# Row 2
fig.add_trace(go.Indicator(
    mode="number", value=recall * 100,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#9b59b6'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=net_savings,
    number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 36, 'color': '#27ae60'}}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=roi_pct,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#f39c12'}}
), row=2, col=3)

fig.update_layout(
    height=400,
    title='<b>Executive Summary Dashboard</b>',
    paper_bgcolor='#f8f9fa'
)
fig.show()

In [ ]:
# Print final recommendations
print("\n" + "=" * 80)
print("EXECUTIVE SUMMARY & RECOMMENDATIONS")
print("=" * 80)

print(f"""
KEY FINDINGS
{'='*60}

1. PROBLEM SCOPE
   - {late_rate:.1f}% of orders experience late delivery
   - Estimated annual cost: ${scenario1_total:,.0f}

2. MODEL PERFORMANCE
   - Catches {recall*100:.0f}% of late deliveries before they happen
   - {accuracy*100:.0f}% overall accuracy
   - False alarm rate: {fpr_rate:.1f}%

3. BUSINESS IMPACT
   - Estimated annual savings: ${net_savings:,.0f}
   - ROI: {roi_pct:.0f}% cost reduction
   - Payback: Immediate (operational cost only)

RECOMMENDATIONS
{'='*60}

1. DEPLOY MODEL
   - Integrate with order management system
   - Real-time scoring at order creation
   - Risk tier assignment for each order

2. OPERATIONAL ACTIONS
   - Critical risk: Automatic shipping upgrade
   - High risk: Proactive customer notification
   - Medium risk: Enhanced tracking & monitoring

3. CONTINUOUS IMPROVEMENT
   - Monthly model retraining with new data
   - A/B testing intervention strategies
   - Track actual vs predicted performance

NEXT STEPS
{'='*60}

1. Pilot program: Deploy on 10% of orders
2. Measure: Track intervention effectiveness
3. Scale: Roll out to full order volume
4. Optimize: Fine-tune thresholds based on results

""")

In [ ]:
# Final presentation slide summary
print("\n" + "=" * 80)
print("10-MINUTE PRESENTATION SUMMARY")
print("=" * 80)

print("""
SLIDE 1: THE PROBLEM (1 min)
- "We have a late delivery problem affecting customer satisfaction"
- Show: {late_rate:.1f}% late delivery rate, ${scenario1_total:,.0f} annual cost

SLIDE 2: THE SOLUTION (2 min)
- "ML model predicts late deliveries BEFORE they happen"
- Features used: shipping mode, scheduled days, customer history
- Leakage prevention: excluded post-delivery data

SLIDE 3: MODEL RESULTS (2 min)
- {recall*100:.0f}% of late deliveries caught
- {accuracy*100:.0f}% overall accuracy
- Show: confusion matrix, ROC curve

SLIDE 4: BUSINESS IMPACT (3 min)
- ${net_savings:,.0f} annual savings ({roi_pct:.0f}% reduction)
- Risk-based intervention strategy
- Show: cost comparison, savings dashboard

SLIDE 5: NEXT STEPS (2 min)
- Deploy to production
- Pilot on 10% of orders
- Continuous monitoring and improvement

""".format(
    late_rate=late_rate,
    scenario1_total=scenario1_total,
    recall=recall,
    accuracy=accuracy,
    net_savings=net_savings,
    roi_pct=roi_pct
))